# Preprocesamiento: Normalización de Tarifas y Construcción del Panel

## Contexto del problema

El dataset histórico de gastos de viaje de FIRA cubre registros desde **enero 2018 hasta abril 2026** (501,716 renglones). Antes de poder explorar o modelar estos datos, es necesario resolver dos problemas estructurales:

1. **Los montos históricos no son comparables entre sí** porque las tarifas máximas de viáticos cambiaron en tres momentos distintos. Un gasto de $1,000 en 2019 representaba el 80% de la cuota vigente entonces; el mismo monto en 2025 representa apenas el 37% de la cuota actual. Sin normalizar, el modelo aprendería diferencias de tarifa, no diferencias de comportamiento.

2. **La granularidad del dataset no es la correcta para el modelo**. El objetivo es predecir el gasto mensual por centro gestor y partida presupuestal. El dataset crudo tiene un renglón por cada concepto de gasto dentro de un viaje (gasolina, hotel, comida, etc.), lo que no corresponde a la unidad de planeación presupuestal.

Este notebook resuelve ambos problemas:

```
Dataset crudo (renglón por concepto de viaje)
        ↓  PASO 1: Normalización de tarifas
Dataset con Monto Gasto Normalizado
        ↓  PASO 2: Mapeo de centro de costo → centro gestor
        ↓  PASO 3: Agregación mensual + panel balanceado
Panel: Centro Gestor × Partida Presupuestal × Mes  ← insumo para EDA y modelo
```

---
## 0. Imports y configuración

In [47]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)

DATA_PATH = 'historico_SAP.parquet'

---
## 1. Carga y revisión inicial

In [48]:
df = pd.read_parquet(DATA_PATH)
df = df.rename(columns={'RESTO DE PERSONAL': 'Categoria'})

print(f'Registros totales : {len(df):,}')
print(f'Columnas          : {df.shape[1]}')
print(f'Rango temporal    : {df["Fecha Contable"].min().date()} → {df["Fecha Contable"].max().date()}')
print(f'Viajes únicos     : {df[["Clave Empleado","Viaje"]].drop_duplicates().shape[0]:,}')
print( '                   (viaje único = par Clave Empleado + Viaje;')
print( '                    el mismo número de viaje puede repetirse en empleados distintos)')
print(f'Empleados únicos  : {df["Clave Empleado"].nunique():,}')
df.head(3)

Registros totales : 501,716
Columnas          : 13
Rango temporal    : 2018-01-02 → 2026-04-30
Viajes únicos     : 147,136
                   (viaje único = par Clave Empleado + Viaje;
                    el mismo número de viaje puede repetirse en empleados distintos)
Empleados únicos  : 1,567


,Clave Empleado,Viaje,Concepto,Fecha Contable,Partida Presupuestal,Categoria,Grupo Tipo Viaje,Clave Centro Costo,Fecha inicio,Fecha Final,Monto Gasto,Periodo,Zona
0,"9,146.00","3,909.00",Peajes con CFDI,2019-08-12,37204,FUNCIONARIO,GC,10000000,2019-08-05,2019-08-06,79.00,1,Zona B
1,"79,292.00","3,504.00",Peajes con CFDI,2018-01-19,37201,RESTO DE PERSONAL,GC,19030200,2018-01-10,2018-01-10,395.99,1,Zona B
2,"79,292.00","3,675.00",Peajes con CFDI,2018-03-08,37201,RESTO DE PERSONAL,GC,19030200,2018-02-28,2018-02-28,322.00,1,Zona B


In [49]:
col_desc = {
    'Clave Empleado'       : 'ID del empleado en SAP',
    'Viaje'                : 'ID del viaje (empleado + viaje = viaje único)',
    'Concepto'             : 'Descripción del tipo de gasto (gasolina, hotel, cuota, etc.)',
    'Fecha Contable'       : 'Fecha en que el gasto impactó contablemente al presupuesto',
    'Partida Presupuestal' : 'Partida del clasificador por objeto del gasto',
    'Categoria'            : 'FUNCIONARIO o RESTO DE PERSONAL — define la tarifa aplicable y a que presupuesto afecta el gasto',
    'Grupo Tipo Viaje'     : 'GC = Gasto Corriente, CAP = Capacitación',
    'Clave Centro Costo'   : 'Centro de costo SAP (nivel operativo detalle)',
    'Fecha inicio'         : 'Primer día del viaje',
    'Fecha Final'          : 'Último día del viaje',
    'Documento Contable'   : 'Folio del documento en SAP',
    'Monto Gasto'          : 'Monto en pesos (negativo = reversa contable)',
    'Periodo'              : 'Periodo tarifario (1, 2 o 3) — resuelto desde SAP',
    'Zona'                 : 'Zona tarifaria del destino (A/B) — resuelta desde SAP',
}
pd.DataFrame({'Descripción': col_desc}).rename_axis('Columna')

,Descripción
Columna,
Categoria,FUNCIONARIO o RESTO DE PERSONAL — define la ta...
Clave Centro Costo,Centro de costo SAP (nivel operativo detalle)
Clave Empleado,ID del empleado en SAP
Concepto,"Descripción del tipo de gasto (gasolina, hotel..."
Documento Contable,Folio del documento en SAP
Fecha Contable,Fecha en que el gasto impactó contablemente al...
Fecha Final,Último día del viaje
Fecha inicio,Primer día del viaje
Grupo Tipo Viaje,"GC = Gasto Corriente, CAP = Capacitación"


In [50]:
# Calidad de datos
print('Valores faltantes:')
nulos = df.isnull().sum()
print(pd.concat([nulos[nulos>0].rename('Nulos'), (nulos[nulos>0]/len(df)*100).round(3).rename('%')], axis=1).to_string())

print(f'\nMontos negativos: {(df["Monto Gasto"]<0).sum()} ({(df["Monto Gasto"]<0).mean()*100:.3f}%) — reversas contables')
print(f'\nRegistros por Periodo:')
print(df['Periodo'].value_counts().sort_index().to_string())
print(f'\nConceptos:')
print(df['Concepto'].value_counts().to_string())

Valores faltantes:
                Nulos    %
Clave Empleado     33 0.01
Viaje              33 0.01
Fecha inicio      296 0.06
Fecha Final       296 0.06

Montos negativos: 91 (0.018%) — reversas contables

Registros por Periodo:
Periodo
1    323032
2    127553
3     51131

Conceptos:
Concepto
Cuota sin comprobar               146014
Gasolina Auto c/financiamiento     99796
Impuestos Alim-Hosp                93589
Alimentación sin Pernocta          60844
Peajes con CFDI                    32975
Alimentación-Hospedaje             32860
Peajes sin CFDI                     8258
Taxis sin CFDI                      4740
Estacionamiento sin CFDI            4689
Gasolina Vehículo Particular        3624
Taxis con CFDI                      3376
Transportación Terrestre            2921
MM Avión                            2675
Estacionamiento con CFDI            2059
Gasolina Vehículo Oficial           1632
Cuota Uso Vehículo Particular       1262
Servicio de Tintorería               239
Transpor

---
## PASO 1 – Normalización de tarifas

### 1.1 ¿Por qué normalizar?

Las tarifas máximas de viáticos (partidas 375xx) cambiaron en tres períodos. Los campos `Periodo` y `Zona` ya vienen resueltos desde SAP para cada registro:

| Periodo | Vigencia | Cambio |
|:---:|---|---|
| 1 | 01/ene/2018 – 20/jul/2023 | Tarifa base con Zona A y Zona B |
| 2 | 21/jul/2023 – 29/jun/2025 | Mismas tarifas; se amplían destinos en Zona A |
| 3 | 30/jun/2025 – presente | Tarifa única más alta; desaparece la distinción de zonas |

Sin normalizar, el modelo aprende diferencias de tarifa institucional entre años, no diferencias de comportamiento de gasto. La normalización convierte todos los montos al valor equivalente que tendrían con las tarifas actuales (Periodo 3).

### 1.2 Tabla de tarifas históricas

Las tarifas regulan tres tipos de cuota diaria:
- **`hosp_alim`**: cuota máxima cuando hay pernocta (alimentación + hospedaje)
- **`alim_sp`**: cuota máxima cuando no hay pernocta (alimentación sin pernocta)
- **`cuota`**: cuota sin comprobar — monto fijo diario sin necesidad de factura

In [51]:
# Clave: (periodo, zona, categoria) → (hosp_alim/día, alim_sp/día, cuota/día)
TARIFAS = {
    (1, 'Zona A', 'FUNCIONARIO')       : (2290, 647, 230),
    (1, 'Zona B', 'FUNCIONARIO')       : (1810, 600, 230),
    (1, 'Zona A', 'RESTO DE PERSONAL') : (1510, 600, 170),
    (1, 'Zona B', 'RESTO DE PERSONAL') : (1290, 600, 170),
    (2, 'Zona A', 'FUNCIONARIO')       : (2290, 647, 230),
    (2, 'Zona B', 'FUNCIONARIO')       : (1810, 600, 230),
    (2, 'Zona A', 'RESTO DE PERSONAL') : (1510, 600, 170),
    (2, 'Zona B', 'RESTO DE PERSONAL') : (1290, 600, 170),
    (3, 'Zona A', 'FUNCIONARIO')       : (2704, 647, 260),
    (3, 'Zona A', 'RESTO DE PERSONAL') : (1635, 620, 230),
}

TARIFAS_P3 = {
    'FUNCIONARIO'       : {'hosp_alim': 2704, 'alim_sp': 647, 'cuota': 260},
    'RESTO DE PERSONAL' : {'hosp_alim': 1635, 'alim_sp': 620, 'cuota': 230},
}

rows = []
for (per, zona, cat), (ha, asp, cuota) in TARIFAS.items():
    rows.append({'Periodo': per, 'Zona': zona, 'Categoría': cat,
                 'Hosp+Alim/día': ha, 'Alim sin pernocta/día': asp, 'Cuota/día': cuota})

pd.DataFrame(rows).sort_values(['Periodo','Zona','Categoría']).style \
    .format({'Hosp+Alim/día':'${:,.0f}','Alim sin pernocta/día':'${:,.0f}','Cuota/día':'${:,.0f}'}) \
    .set_caption('Tarifas máximas de viáticos por periodo')

,Periodo,Zona,Categoría,Hosp+Alim/día,Alim sin pernocta/día,Cuota/día
0,1,Zona A,FUNCIONARIO,"$2,290",$647,$230
2,1,Zona A,RESTO DE PERSONAL,"$1,510",$600,$170
1,1,Zona B,FUNCIONARIO,"$1,810",$600,$230
3,1,Zona B,RESTO DE PERSONAL,"$1,290",$600,$170
4,2,Zona A,FUNCIONARIO,"$2,290",$647,$230
6,2,Zona A,RESTO DE PERSONAL,"$1,510",$600,$170
5,2,Zona B,FUNCIONARIO,"$1,810",$600,$230
7,2,Zona B,RESTO DE PERSONAL,"$1,290",$600,$170
8,3,Zona A,FUNCIONARIO,"$2,704",$647,$260
9,3,Zona A,RESTO DE PERSONAL,"$1,635",$620,$230


### 1.3 Lógica de normalización

```
¿Periodo 3?              → SIN CAMBIO  (ya está en tarifa vigente)
¿Monto negativo?         → SIN CAMBIO  (reversa contable; preservar impacto original)
¿Cuota sin comprobar?    → cuota_P3 × días  (cuota completa; no hay gasto real que proporcionar)
¿Alim-Hospedaje?         → (monto / cuota_hist) × cuota_P3
¿Alim sin Pernocta?      → (monto / cuota_hist) × cuota_P3
Cualquier otro concepto  → SIN CAMBIO  (gasolina, peajes, taxis: no dependen de tarifas)
```

**¿Por qué proporciones para Alim-Hosp y Alim sin Pernocta?** Son gastos comprobables con factura. Lo que importa para el modelo es qué fracción de la cuota ejerció el empleado. Si en 2019 gastó el 60% de su cuota, con las tarifas de hoy ese mismo comportamiento equivale al 60% de la cuota actual. La proporción preserva el patrón.

**¿Por qué cuota completa para Cuota sin comprobar?** No hay un gasto real: el empleado recibe la cuota íntegra por día. Se sustituye por lo que le correspondería hoy.

**¿Por qué sin cambio para montos negativos?** Son reversas contables. Deben conservar el valor con el que impactaron originalmente al presupuesto.

**Ejemplos numéricos:**

| Concepto | Original | Cálculo | Normalizado |
|---|---|---|---|
| Cuota sin comprobar (Funcionario, Zona A, P1, 2 días) | $460 | 260 × 2 días | **$520** |
| Alim-Hospedaje (Funcionario, Zona B, P1, 2 días) | $1,244.54 | (1244.54/3620) × 5408 | **$1,859.25** |
| Monto negativo | −$228.76 | sin cambio | **−$228.76** |

In [52]:
df['dias'] = (df['Fecha Final'] - df['Fecha inicio']).dt.days + 1

per, zona, cat = df['Periodo'], df['Zona'], df['Categoria']

df['tarifa_hist_hosp_alim'] = [TARIFAS.get((p,z,c),(np.nan,)*3)[0] for p,z,c in zip(per,zona,cat)]
df['tarifa_hist_alim_sp']   = [TARIFAS.get((p,z,c),(np.nan,)*3)[1] for p,z,c in zip(per,zona,cat)]

df['tarifa_p3_hosp_alim'] = cat.map({k: v['hosp_alim'] for k,v in TARIFAS_P3.items()})
df['tarifa_p3_alim_sp']   = cat.map({k: v['alim_sp']   for k,v in TARIFAS_P3.items()})
df['tarifa_p3_cuota']     = cat.map({k: v['cuota']      for k,v in TARIFAS_P3.items()})

df['cuota_hist_hosp_alim'] = df['tarifa_hist_hosp_alim'] * df['dias']
df['cuota_hist_alim_sp']   = df['tarifa_hist_alim_sp']   * df['dias']
df['cuota_p3_hosp_alim']   = df['tarifa_p3_hosp_alim']   * df['dias']
df['cuota_p3_alim_sp']     = df['tarifa_p3_alim_sp']     * df['dias']
df['cuota_p3_cuota']       = df['tarifa_p3_cuota']       * df['dias']

df['pct_hosp_alim'] = np.where(df['cuota_hist_hosp_alim']!=0,
                               df['Monto Gasto']/df['cuota_hist_hosp_alim'], np.nan)
df['pct_alim_sp']   = np.where(df['cuota_hist_alim_sp']!=0,
                               df['Monto Gasto']/df['cuota_hist_alim_sp'],   np.nan)

print('Variables de tarifa y cuota calculadas. Muestra:')
df[
    df['Concepto'].isin(['Cuota sin comprobar', 'Alimentación-Hospedaje', 'Alimentación sin Pernocta']) &
    (df['Periodo'] != 3) &
    (df['Monto Gasto'] > 0)
][['Concepto', 'Monto Gasto', 'dias', 'Categoria', 'Zona', 'Periodo',
   'cuota_hist_hosp_alim', 'cuota_p3_cuota']].dropna(subset=['dias']).head(4)

Variables de tarifa y cuota calculadas. Muestra:


,Concepto,Monto Gasto,dias,Categoria,Zona,Periodo,cuota_hist_hosp_alim,cuota_p3_cuota
168161,Cuota sin comprobar,232.00,2.00,FUNCIONARIO,Zona A,1,"4,580.00",520.00
168162,Cuota sin comprobar,116.00,1.00,FUNCIONARIO,Zona B,1,"1,810.00",260.00
168163,Cuota sin comprobar,232.00,2.00,FUNCIONARIO,Zona B,1,"3,620.00",520.00
168164,Alimentación-Hospedaje,"1,244.54",2.00,FUNCIONARIO,Zona B,1,"3,620.00",520.00


In [53]:
concepto = df['Concepto']
monto    = df['Monto Gasto']
es_p3    = df['Periodo'] == 3
es_neg   = monto < 0

df['Monto Gasto Normalizado'] = np.where(
    es_p3 | es_neg,
    monto,                                                          # Reglas 1 y 2: sin cambio
    np.where(
        concepto == 'Cuota sin comprobar',
        df['cuota_p3_cuota'],                                       # Regla 3: cuota completa actual
        np.where(
            concepto == 'Alimentación-Hospedaje',
            df['pct_hosp_alim'] * df['cuota_p3_hosp_alim'],        # Regla 4: proporción × cuota P3
            np.where(
                concepto == 'Alimentación sin Pernocta',
                df['pct_alim_sp'] * df['cuota_p3_alim_sp'],        # Regla 4: proporción × cuota P3
                monto                                               # Regla 5: resto sin cambio
            )
        )
    )
)

CONCEPTOS_NORM = {'Alimentación-Hospedaje', 'Alimentación sin Pernocta', 'Cuota sin comprobar'}
df['fue_normalizado'] = ~es_p3 & ~es_neg & concepto.isin(CONCEPTOS_NORM)

print('Normalización aplicada.')

Normalización aplicada.


In [54]:
# Verificaciones de integridad
print('── Verificaciones ──────────────────────────────────────────────────────')
print(f'✓ Dif. máx. en Periodo 3   : {(df[es_p3]["Monto Gasto"]-df[es_p3]["Monto Gasto Normalizado"]).abs().max():.4f}  (debe ser 0.0)')
print(f'✓ Dif. máx. en negativos   : {(df[es_neg]["Monto Gasto"]-df[es_neg]["Monto Gasto Normalizado"]).abs().max():.4f}  (debe ser 0.0)')
print(f'✓ NaN en Monto Normalizado : {df["Monto Gasto Normalizado"].isna().sum()}  (debe ser 0)')

print('\n── Efecto por concepto [periodos 1-2, montos positivos] ────────────────')
for c in ['Cuota sin comprobar', 'Alimentación-Hospedaje', 'Alimentación sin Pernocta']:
    sub = df[(concepto==c) & ~es_p3 & ~es_neg]
    orig = sub['Monto Gasto'].mean()
    norm = sub['Monto Gasto Normalizado'].mean()
    print(f'  {c:<30} orig=${orig:>7,.0f}  norm=${norm:>7,.0f}  cambio={(norm/orig-1)*100:+.1f}%')

print('\n── Registros por tipo de tratamiento ───────────────────────────────────')
resumen = pd.DataFrame([
    {'Tratamiento': 'Periodo 3 (tarifa vigente)',         'Regla': 'Sin cambio',         'n': es_p3.sum()},
    {'Tratamiento': 'Monto negativo (reversa contable)',  'Regla': 'Sin cambio',         'n': (~es_p3 & es_neg & concepto.isin(CONCEPTOS_NORM)).sum()},
    {'Tratamiento': 'Cuota sin comprobar',                'Regla': 'cuota_P3 × días',   'n': (~es_p3 & ~es_neg & (concepto=='Cuota sin comprobar')).sum()},
    {'Tratamiento': 'Alimentación-Hospedaje',             'Regla': 'pct × cuota_P3',    'n': (~es_p3 & ~es_neg & (concepto=='Alimentación-Hospedaje')).sum()},
    {'Tratamiento': 'Alimentación sin Pernocta',          'Regla': 'pct × cuota_P3',    'n': (~es_p3 & ~es_neg & (concepto=='Alimentación sin Pernocta')).sum()},
    {'Tratamiento': 'Otros conceptos',                    'Regla': 'Sin cambio',         'n': (~es_p3 & ~concepto.isin(CONCEPTOS_NORM)).sum()},
])
resumen['%'] = (resumen['n']/len(df)*100).round(1).astype(str)+'%'
resumen

── Verificaciones ──────────────────────────────────────────────────────
✓ Dif. máx. en Periodo 3   : 0.0000  (debe ser 0.0)
✓ Dif. máx. en negativos   : 0.0000  (debe ser 0.0)
✓ NaN en Monto Normalizado : 0  (debe ser 0)

── Efecto por concepto [periodos 1-2, montos positivos] ────────────────
  Cuota sin comprobar            orig=$    183  norm=$    335  cambio=+83.5%
  Alimentación-Hospedaje         orig=$  2,413  norm=$  2,903  cambio=+20.3%
  Alimentación sin Pernocta      orig=$    369  norm=$    381  cambio=+3.3%

── Registros por tipo de tratamiento ───────────────────────────────────


,Tratamiento,Regla,n,%
0,Periodo 3 (tarifa vigente),Sin cambio,51131,10.2%
1,Monto negativo (reversa contable),Sin cambio,32,0.0%
2,Cuota sin comprobar,cuota_P3 × días,130684,26.0%
3,Alimentación-Hospedaje,pct × cuota_P3,29618,5.9%
4,Alimentación sin Pernocta,pct × cuota_P3,54219,10.8%
5,Otros conceptos,Sin cambio,236032,47.0%


### Nota sobre la Cuota sin comprobar (+83.5%)

El cambio más pronunciado en la normalización corresponde a la **Cuota sin comprobar**,
que pasa de un promedio de $183 a $335 (+83.5%). Este incremento refleja dos factores combinados:

1. **Las tarifas del Periodo 3 son más altas** — la cuota diaria subió de $170–$230
   a $230–$260 dependiendo de la categoría.

2. **Un cambio de práctica operativa:** históricamente existía la práctica de comprobar
   un día menos de cuota sin comprobar por viaje (por ejemplo, en un viaje de 3 días
   se registraban solo 2 días de cuota). Esta práctica fue eliminada a partir de los
   cambios de tarifa del Periodo 3 y actualizaciones al reglamento de viáticos,
   de modo que hoy la cuota se aplica por todos los días del viaje.

Por esta razón, la normalización de la cuota sin comprobar **no usa proporción**
(a diferencia de alimentación y hospedaje): aplicar la proporción histórica
perpetuaría una práctica que ya no existe. En cambio, se sustituye directamente
por la cuota completa que correspondería con las tarifas actuales:

$$\text{Normalizado} = \text{tarifa\_P3\_cuota} \times \text{días}$$

Esto hace que el modelo aprenda el patrón de gasto **tal como opera hoy**,
no como operaba antes del cambio reglamentario.

---
## PASO 2 – Mapeo a Centro Gestor

### 2.1 Catálogo de centros gestores (embebido)

El dataset tiene `Clave Centro Costo` (238 valores — nivel operativo). El modelo opera al nivel de **Centro Gestor** (36 valores — unidad de control presupuestal).

In [55]:
# Catálogo Centro de Costo → (Centro Gestor, Nombre del Centro Gestor)
# Fuente: Catálogo de Unidades Administrativas FIRA
# Los 8 CCos marcados como 'histórico' no están en el catálogo actual
# (centros dados de baja o reclasificados) — se resuelven por prefijo de 4 dígitos
CATALOGO = {
    10000000: (10000000, 'Dirección General'),
    10020000: (10020000, 'Unidad de Análisis y Administración Integral de Riesgos'),
    10020100: (10020000, 'Unidad de Análisis y Administración Integral de Riesgos'),
    10020200: (10020000, 'Unidad de Análisis y Administración Integral de Riesgos'),
    10040000: (10040000, 'Contraloría Interna'),
    10040100: (10040000, 'Contraloría Interna'),
    10040200: (10040000, 'Contraloría Interna'),
    10050000: (10050000, 'Dirección de Auditoría Interna'),
    10050100: (10050000, 'Dirección de Auditoría Interna'),
    10050200: (10050000, 'Dirección de Auditoría Interna'),
    10060100: (10060100, 'Subdirección de Proyectos y Estrategia'),
    10060101: (10060100, 'Subdirección de Proyectos y Estrategia'),
    10060102: (10060100, 'Subdirección de Proyectos y Estrategia'),
    10060103: (10060100, 'Subdirección de Proyectos y Estrategia'),
    10060104: (10060100, 'Subdirección de Proyectos y Estrategia'),
    10060105: (10060100, 'Subdirección de Proyectos y Estrategia'),
    11000000: (11000000, 'Dirección General Adjunta de Inteligencia Sectorial'),
    11060000: (11060000, 'Dirección de Investigación y Evaluación Económica y Sectorial'),
    11060100: (11060000, 'Dirección de Investigación y Evaluación Económica y Sectorial'),
    11060200: (11060000, 'Dirección de Investigación y Evaluación Económica y Sectorial'),
    11060300: (11060000, 'Dirección de Investigación y Evaluación Económica y Sectorial'),
    11070000: (11070000, 'Dirección de Medio Ambiente, Pesca y Redes de Valor'),
    11070100: (11070000, 'Dirección de Medio Ambiente, Pesca y Redes de Valor'),
    11070200: (11070000, 'Dirección de Medio Ambiente, Pesca y Redes de Valor'),
    12000000: (12000000, 'Dirección General Adjunta de Crédito'),
    12010000: (12010000, 'Dirección de Supervisión y Monitoreo'),
    12010300: (12010000, 'Dirección de Supervisión y Monitoreo'),
    12010400: (12010000, 'Dirección de Supervisión y Monitoreo'),
    12010500: (12010000, 'Dirección de Supervisión y Monitoreo'),
    12020000: (12020000, 'Dirección de Crédito'),
    12020400: (12020000, 'Dirección de Crédito'),
    12020600: (12020000, 'Dirección de Crédito'),
    12020700: (12020000, 'Dirección de Crédito'),
    12020800: (12020000, 'Dirección de Crédito'),
    12090000: (12090000, 'Dirección de Enlace con Intermediarios Financieros'),
    12090100: (12090000, 'Dirección de Enlace con Intermediarios Financieros'),
    12090200: (12090000, 'Dirección de Enlace con Intermediarios Financieros'),
    13000000: (13000000, 'Dirección General Adjunta de Finanzas'),
    13010000: (13010000, 'Dirección de Finanzas y Planeación Corporativa'),
    13010100: (13010000, 'Dirección de Finanzas y Planeación Corporativa'),
    13010101: (13010000, 'Dirección de Finanzas y Planeación Corporativa'),
    13010102: (13010000, 'Dirección de Finanzas y Planeación Corporativa'),
    13010200: (13010000, 'Dirección de Finanzas y Planeación Corporativa'),
    13010201: (13010000, 'Dirección de Finanzas y Planeación Corporativa'),
    13010203: (13010000, 'Dirección de Finanzas y Planeación Corporativa'),
    13010300: (13010000, 'Dirección de Finanzas y Planeación Corporativa'),
    13020000: (13020000, 'Dirección de Tesorería'),
    13020100: (13020000, 'Dirección de Tesorería'),
    13020300: (13020000, 'Dirección de Tesorería'),
    14000000: (14000000, 'Dirección General Adjunta de Promoción de Negocios'),
    14130000: (14130000, 'Dirección de Programas y Proyectos'),
    14130100: (14130000, 'Dirección de Programas y Proyectos'),
    14130200: (14130000, 'Dirección de Programas y Proyectos'),
    14140000: (14140000, 'Dirección de Atención Corporativa a Intermediarios Financieros'),
    14140100: (14140000, 'Dirección de Atención Corporativa a Intermediarios Financieros'),
    14140200: (14140000, 'Dirección de Atención Corporativa a Intermediarios Financieros'),
    14150000: (14150000, 'Dirección de Desarrollo y Promoción de Productos y Servicios'),
    14150100: (14150000, 'Dirección de Desarrollo y Promoción de Productos y Servicios'),
    14150200: (14150000, 'Dirección de Desarrollo y Promoción de Productos y Servicios'),
    14150300: (14150000, 'Dirección de Desarrollo y Promoción de Productos y Servicios'),
    15000000: (15000000, 'Dirección General Adjunta de Sistemas y Operaciones'),
    15010000: (15010000, 'Dirección de Sistemas'),
    15010100: (15010000, 'Dirección de Sistemas'),
    15010200: (15010000, 'Dirección de Sistemas'),
    15010300: (15010000, 'Dirección de Sistemas'),
    15010400: (15010000, 'Dirección de Sistemas'),
    15030000: (15030000, 'Dirección de Control de Operaciones'),
    15030100: (15030000, 'Dirección de Control de Operaciones'),
    15030200: (15030000, 'Dirección de Control de Operaciones'),
    15030300: (15030000, 'Dirección de Control de Operaciones'),
    15030700: (15030000, 'Dirección de Control de Operaciones'),
    16000000: (16000000, 'Dirección General Adjunta de Administración y Jurídica'),
    16010000: (16010000, 'Dirección Jurídica y de Recuperación'),
    16010200: (16010000, 'Dirección Jurídica y de Recuperación'),
    16010300: (16010000, 'Dirección Jurídica y de Recuperación'),
    16010400: (16010000, 'Dirección Jurídica y de Recuperación'),
    16010500: (16010000, 'Dirección Jurídica y de Recuperación'),
    16020000: (16020000, 'Dirección de Administración'),
    16020300: (16020000, 'Dirección de Administración'),
    16020303: (16020000, 'Dirección de Administración'),
    16020304: (16020000, 'Dirección de Administración'),
    16020390: (16020000, 'Dirección de Administración'),  # histórico — prefijo 1602
    16021300: (16020000, 'Dirección de Administración'),
    16021308: (16020000, 'Dirección de Administración'),
    16021309: (16020000, 'Dirección de Administración'),
    16021310: (16020000, 'Dirección de Administración'),
    16021311: (16020000, 'Dirección de Administración'),
    16021400: (16020000, 'Dirección de Administración'),
    16021500: (16020000, 'Dirección de Administración'),
    17000000: (17000000, 'Órgano Interno de Control'),
    17010000: (17010000, 'Área de Auditoría para Desarrollo y Mejora de la Gestión Pública'),
    17010300: (17010000, 'Área de Auditoría para Desarrollo y Mejora de la Gestión Pública'),  # histórico — prefijo 1701
    17010500: (17010000, 'Área de Auditoría para Desarrollo y Mejora de la Gestión Pública'),
    17010600: (17010000, 'Área de Auditoría para Desarrollo y Mejora de la Gestión Pública'),
    17020000: (17020000, 'Área de Auditoría Interna'),
    17020600: (17020000, 'Área de Auditoría Interna'),
    17020700: (17020000, 'Área de Auditoría Interna'),
    17020800: (17020000, 'Área de Auditoría Interna'),
    17020900: (17020000, 'Área de Auditoría Interna'),
    17030000: (17030000, 'Área de Responsabilidades'),
    17030200: (17030000, 'Área de Responsabilidades'),
    17100000: (17100000, 'Área de Quejas'),
    19000000: (19000000, 'Dirección General Adjunta de Coordinación de Regionales'),
    19010000: (19010000, 'Dirección Regional del Noroeste'),
    19010100: (19010000, 'Dirección Regional del Noroeste'),
    19010200: (19010000, 'Dirección Regional del Noroeste'),
    19010201: (19010000, 'Dirección Regional del Noroeste'),
    19010202: (19010000, 'Dirección Regional del Noroeste'),
    19010203: (19010000, 'Dirección Regional del Noroeste'),
    19010204: (19010000, 'Dirección Regional del Noroeste'),
    19010205: (19010000, 'Dirección Regional del Noroeste'),
    19010300: (19010000, 'Dirección Regional del Noroeste'),
    19010301: (19010000, 'Dirección Regional del Noroeste'),
    19010302: (19010000, 'Dirección Regional del Noroeste'),
    19010303: (19010000, 'Dirección Regional del Noroeste'),
    19010304: (19010000, 'Dirección Regional del Noroeste'),
    19010305: (19010000, 'Dirección Regional del Noroeste'),
    19010400: (19010000, 'Dirección Regional del Noroeste'),
    19010401: (19010000, 'Dirección Regional del Noroeste'),
    19010402: (19010000, 'Dirección Regional del Noroeste'),
    19010403: (19010000, 'Dirección Regional del Noroeste'),
    19010500: (19010000, 'Dirección Regional del Noroeste'),
    19010501: (19010000, 'Dirección Regional del Noroeste'),
    19020000: (19020000, 'Dirección Regional del Norte'),
    19020100: (19020000, 'Dirección Regional del Norte'),
    19020101: (19020000, 'Dirección Regional del Norte'),  # histórico — prefijo 1902
    19020200: (19020000, 'Dirección Regional del Norte'),
    19020201: (19020000, 'Dirección Regional del Norte'),
    19020202: (19020000, 'Dirección Regional del Norte'),
    19020300: (19020000, 'Dirección Regional del Norte'),
    19020301: (19020000, 'Dirección Regional del Norte'),
    19020302: (19020000, 'Dirección Regional del Norte'),
    19020400: (19020000, 'Dirección Regional del Norte'),
    19020401: (19020000, 'Dirección Regional del Norte'),
    19020402: (19020000, 'Dirección Regional del Norte'),
    19020404: (19020000, 'Dirección Regional del Norte'),
    19020500: (19020000, 'Dirección Regional del Norte'),
    19020501: (19020000, 'Dirección Regional del Norte'),
    19020502: (19020000, 'Dirección Regional del Norte'),
    19020600: (19020000, 'Dirección Regional del Norte'),
    19020601: (19020000, 'Dirección Regional del Norte'),
    19020602: (19020000, 'Dirección Regional del Norte'),
    19020603: (19020000, 'Dirección Regional del Norte'),
    19020604: (19020000, 'Dirección Regional del Norte'),
    19020605: (19020000, 'Dirección Regional del Norte'),
    19020606: (19020000, 'Dirección Regional del Norte'),
    19030000: (19030000, 'Dirección Regional de Occidente'),
    19030100: (19030000, 'Dirección Regional de Occidente'),
    19030101: (19030000, 'Dirección Regional de Occidente'),  # histórico — prefijo 1903
    19030102: (19030000, 'Dirección Regional de Occidente'),  # histórico — prefijo 1903
    19030200: (19030000, 'Dirección Regional de Occidente'),
    19030201: (19030000, 'Dirección Regional de Occidente'),
    19030202: (19030000, 'Dirección Regional de Occidente'),
    19030203: (19030000, 'Dirección Regional de Occidente'),
    19030204: (19030000, 'Dirección Regional de Occidente'),
    19030205: (19030000, 'Dirección Regional de Occidente'),
    19030206: (19030000, 'Dirección Regional de Occidente'),
    19030207: (19030000, 'Dirección Regional de Occidente'),
    19030300: (19030000, 'Dirección Regional de Occidente'),
    19030301: (19030000, 'Dirección Regional de Occidente'),
    19030302: (19030000, 'Dirección Regional de Occidente'),
    19030400: (19030000, 'Dirección Regional de Occidente'),
    19030500: (19030000, 'Dirección Regional de Occidente'),
    19030501: (19030000, 'Dirección Regional de Occidente'),
    19030502: (19030000, 'Dirección Regional de Occidente'),
    19030503: (19030000, 'Dirección Regional de Occidente'),
    19030504: (19030000, 'Dirección Regional de Occidente'),
    19030600: (19030000, 'Dirección Regional de Occidente'),
    19030601: (19030000, 'Dirección Regional de Occidente'),
    19030602: (19030000, 'Dirección Regional de Occidente'),
    19030603: (19030000, 'Dirección Regional de Occidente'),
    19030604: (19030000, 'Dirección Regional de Occidente'),
    19030605: (19030000, 'Dirección Regional de Occidente'),
    19030700: (19030000, 'Dirección Regional de Occidente'),
    19030701: (19030000, 'Dirección Regional de Occidente'),
    19030702: (19030000, 'Dirección Regional de Occidente'),
    19030800: (19030000, 'Dirección Regional de Occidente'),
    19030801: (19030000, 'Dirección Regional de Occidente'),
    19030802: (19030000, 'Dirección Regional de Occidente'),
    19030900: (19030000, 'Dirección Regional de Occidente'),
    19031000: (19030000, 'Dirección Regional de Occidente'),
    19040000: (19040000, 'Dirección Regional del Sur'),
    19040100: (19040000, 'Dirección Regional del Sur'),
    19040101: (19040000, 'Dirección Regional del Sur'),  # histórico — prefijo 1904
    19040200: (19040000, 'Dirección Regional del Sur'),
    19040201: (19040000, 'Dirección Regional del Sur'),
    19040202: (19040000, 'Dirección Regional del Sur'),
    19040203: (19040000, 'Dirección Regional del Sur'),
    19040204: (19040000, 'Dirección Regional del Sur'),
    19040300: (19040000, 'Dirección Regional del Sur'),
    19040400: (19040000, 'Dirección Regional del Sur'),
    19040401: (19040000, 'Dirección Regional del Sur'),
    19040402: (19040000, 'Dirección Regional del Sur'),
    19040403: (19040000, 'Dirección Regional del Sur'),
    19040404: (19040000, 'Dirección Regional del Sur'),
    19040405: (19040000, 'Dirección Regional del Sur'),
    19040406: (19040000, 'Dirección Regional del Sur'),
    19040408: (19040000, 'Dirección Regional del Sur'),
    19040500: (19040000, 'Dirección Regional del Sur'),
    19040501: (19040000, 'Dirección Regional del Sur'),
    19040502: (19040000, 'Dirección Regional del Sur'),  # histórico — prefijo 1904
    19040503: (19040000, 'Dirección Regional del Sur'),
    19040504: (19040000, 'Dirección Regional del Sur'),
    19040506: (19040000, 'Dirección Regional del Sur'),
    19040600: (19040000, 'Dirección Regional del Sur'),
    19040601: (19040000, 'Dirección Regional del Sur'),
    19040602: (19040000, 'Dirección Regional del Sur'),
    19040700: (19040000, 'Dirección Regional del Sur'),
    19040701: (19040000, 'Dirección Regional del Sur'),
    19040702: (19040000, 'Dirección Regional del Sur'),
    19040703: (19040000, 'Dirección Regional del Sur'),
    19040800: (19040000, 'Dirección Regional del Sur'),
    19040801: (19040000, 'Dirección Regional del Sur'),
    19040802: (19040000, 'Dirección Regional del Sur'),
    19040900: (19040000, 'Dirección Regional del Sur'),
    19040901: (19040000, 'Dirección Regional del Sur'),
    19040902: (19040000, 'Dirección Regional del Sur'),
    19040903: (19040000, 'Dirección Regional del Sur'),
    19050000: (19050000, 'Dirección Regional del Sureste'),
    19050100: (19050000, 'Dirección Regional del Sureste'),
    19050101: (19050000, 'Dirección Regional del Sureste'),  # histórico — prefijo 1905
    19050200: (19050000, 'Dirección Regional del Sureste'),
    19050201: (19050000, 'Dirección Regional del Sureste'),
    19050202: (19050000, 'Dirección Regional del Sureste'),
    19050300: (19050000, 'Dirección Regional del Sureste'),
    19050301: (19050000, 'Dirección Regional del Sureste'),
    19050302: (19050000, 'Dirección Regional del Sureste'),
    19050303: (19050000, 'Dirección Regional del Sureste'),
    19050304: (19050000, 'Dirección Regional del Sureste'),
    19050306: (19050000, 'Dirección Regional del Sureste'),
    19050400: (19050000, 'Dirección Regional del Sureste'),
    19050401: (19050000, 'Dirección Regional del Sureste'),
    19050402: (19050000, 'Dirección Regional del Sureste'),
    19050500: (19050000, 'Dirección Regional del Sureste'),
    19050501: (19050000, 'Dirección Regional del Sureste'),
    19050600: (19050000, 'Dirección Regional del Sureste'),
    19050601: (19050000, 'Dirección Regional del Sureste'),
    19050602: (19050000, 'Dirección Regional del Sureste'),
}

# Aplicar mapeo al dataset
df['Centro Gestor']        = df['Clave Centro Costo'].map({k: v[0] for k, v in CATALOGO.items()})
df['Nombre Centro Gestor'] = df['Clave Centro Costo'].map({k: v[1] for k, v in CATALOGO.items()})

# Verificación — debe ser 0 en ambos casos
sin_match = df['Centro Gestor'].isna().sum()
print(f'Centros gestores únicos : {df["Centro Gestor"].nunique()}')
print(f'Registros sin match     : {sin_match}  ← debe ser 0')

# Vista del catálogo de los 36 centros gestores
(pd.DataFrame([{'Centro Gestor': v[0], 'Nombre': v[1]} for v in CATALOGO.values()])
   .drop_duplicates('Centro Gestor')
   .sort_values('Centro Gestor')
   .reset_index(drop=True))

Centros gestores únicos : 36
Registros sin match     : 0  ← debe ser 0


,Centro Gestor,Nombre
0,10000000,Dirección General
1,10020000,Unidad de Análisis y Administración Integral d...
2,10040000,Contraloría Interna
3,10050000,Dirección de Auditoría Interna
4,10060100,Subdirección de Proyectos y Estrategia
5,11000000,Dirección General Adjunta de Inteligencia Sect...
6,11060000,Dirección de Investigación y Evaluación Económ...
7,11070000,"Dirección de Medio Ambiente, Pesca y Redes de ..."
8,12000000,Dirección General Adjunta de Crédito
9,12010000,Dirección de Supervisión y Monitoreo


In [56]:
print("""
El dataset ahora tiene dos columnas nuevas: 'Centro Gestor' (código numérico de la
unidad de presupuesto) y 'Nombre Centro Gestor' (nombre legible). Cada uno de los
238 centros de costo queda asignado a uno de los 36 centros gestores, que son las
unidades de planeación y control presupuestal en FIRA.

Esta asignación es el puente entre el nivel de registro de SAP (centro de costo,
nivel detalle) y el nivel de predicción del modelo (centro gestor × partida × mes).
""")

df[['Clave Centro Costo', 'Centro Gestor', 'Nombre Centro Gestor',
    'Concepto', 'Monto Gasto', 'Monto Gasto Normalizado', 'fue_normalizado']].head(10)


El dataset ahora tiene dos columnas nuevas: 'Centro Gestor' (código numérico de la
unidad de presupuesto) y 'Nombre Centro Gestor' (nombre legible). Cada uno de los
238 centros de costo queda asignado a uno de los 36 centros gestores, que son las
unidades de planeación y control presupuestal en FIRA.

Esta asignación es el puente entre el nivel de registro de SAP (centro de costo,
nivel detalle) y el nivel de predicción del modelo (centro gestor × partida × mes).



,Clave Centro Costo,Centro Gestor,Nombre Centro Gestor,Concepto,Monto Gasto,Monto Gasto Normalizado,fue_normalizado
0,10000000,10000000,Dirección General,Peajes con CFDI,79.00,79.00,False
1,19030200,19030000,Dirección Regional de Occidente,Peajes con CFDI,395.99,395.99,False
2,19030200,19030000,Dirección Regional de Occidente,Peajes con CFDI,322.00,322.00,False
3,19030200,19030000,Dirección Regional de Occidente,Gasolina Auto c/financiamiento,500.00,500.00,False
4,19030200,19030000,Dirección Regional de Occidente,Gasolina Auto c/financiamiento,792.80,792.80,False
5,17010000,17010000,Área de Auditoría para Desarrollo y Mejora de ...,Peajes con CFDI,955.99,955.99,False
6,17010000,17010000,Área de Auditoría para Desarrollo y Mejora de ...,Gasolina Auto c/financiamiento,"1,555.16","1,555.16",False
7,17010000,17010000,Área de Auditoría para Desarrollo y Mejora de ...,Estacionamiento sin CFDI,348.00,348.00,False
8,14130100,14130000,Dirección de Programas y Proyectos,Gasolina Vehículo Particular,991.76,991.76,False
9,14130100,14130000,Dirección de Programas y Proyectos,Transportación Terrestre,562.50,562.50,False


---
## PASO 3 – Agregación y panel balanceado

### 3.1 ¿Por qué esta granularidad?

El objetivo del modelo es predecir el gasto mensual de un centro gestor en una partida presupuestal. Esa es la unidad con la que FIRA planea y controla su presupuesto. La agregación suma todos los `Monto Gasto Normalizado` que comparten el mismo Centro Gestor, Partida y mes de `Fecha Contable`.

> Se usa `Fecha Contable` (no `Fecha inicio`) porque es la fecha en que el gasto impacta al presupuesto en SAP.

### 3.2 Panel balanceado

El dataset agregado solo tiene filas donde hubo gasto. Para el modelo de series de tiempo se necesita un **panel completo**: los meses sin gasto deben aparecer como **$0**, no como datos faltantes.

In [57]:
df['Anio']     = pd.to_datetime(df['Fecha Contable']).dt.year
df['Mes']      = pd.to_datetime(df['Fecha Contable']).dt.month
df['Anio_Mes'] = pd.to_datetime(df['Fecha Contable']).dt.to_period('M')

df_agg = df[df['Centro Gestor'] != 'SIN_MATCH'].copy()

df_mensual = (
    df_agg
    .groupby(['Centro Gestor','Nombre Centro Gestor','Partida Presupuestal','Anio','Mes','Anio_Mes'])
    .agg(
        Gasto_Normalizado = ('Monto Gasto Normalizado', 'sum'),
        Gasto_Original    = ('Monto Gasto',             'sum'),
        Num_Registros     = ('Monto Gasto',             'count'),
        Num_Viajes        = ('Viaje',                   'nunique'),
    )
    .reset_index()
)

print(f'Combinaciones observadas (CG × Partida × Mes): {len(df_mensual):,}')
df_mensual.head(4)

Combinaciones observadas (CG × Partida × Mes): 10,212


,Centro Gestor,Nombre Centro Gestor,Partida Presupuestal,Anio,Mes,Anio_Mes,Gasto_Normalizado,Gasto_Original,Num_Registros,Num_Viajes
0,10000000,Dirección General,26102,2018,2,2018-02,580.10,580.10,1,1
1,10000000,Dirección General,26102,2018,5,2018-05,"1,160.35","1,160.35",2,2
2,10000000,Dirección General,26102,2018,6,2018-06,"1,220.53","1,220.53",1,1
3,10000000,Dirección General,26102,2018,8,2018-08,450.20,450.20,1,1


In [58]:
# Producto cartesiano: todas las combinaciones CG × Partida × Mes del rango histórico
todos_los_meses = pd.period_range(start=df_mensual['Anio_Mes'].min(),
                                   end=df_mensual['Anio_Mes'].max(), freq='M')
combinaciones = df_mensual[['Centro Gestor','Nombre Centro Gestor','Partida Presupuestal']].drop_duplicates()
indice_completo = combinaciones.merge(pd.DataFrame({'Anio_Mes': todos_los_meses}), how='cross')
indice_completo['Anio'] = indice_completo['Anio_Mes'].dt.year
indice_completo['Mes']  = indice_completo['Anio_Mes'].dt.month

# Left join: los meses sin gasto quedan como NaN y se rellenan con 0
df_panel = indice_completo.merge(
    df_mensual,
    on=['Centro Gestor','Nombre Centro Gestor','Partida Presupuestal','Anio','Mes','Anio_Mes'],
    how='left'
)
df_panel['Gasto_Normalizado'] = df_panel['Gasto_Normalizado'].fillna(0)
df_panel['Gasto_Original']    = df_panel['Gasto_Original'].fillna(0)
df_panel['Num_Registros']     = df_panel['Num_Registros'].fillna(0).astype(int)
df_panel['Num_Viajes']        = df_panel['Num_Viajes'].fillna(0).astype(int)
df_panel = df_panel.sort_values(['Centro Gestor','Partida Presupuestal','Anio_Mes']).reset_index(drop=True)

print(f'Panel observado  : {len(df_mensual):,} filas')
print(f'Panel completo   : {len(df_panel):,} filas')
print(f'Meses con $0     : {(df_panel["Gasto_Normalizado"]==0).sum():,} ({(df_panel["Gasto_Normalizado"]==0).mean()*100:.1f}%)')

Panel observado  : 10,212 filas
Panel completo   : 27,200 filas
Meses con $0     : 16,988 (62.5%)


In [59]:
# Validación final: la suma total debe preservarse exactamente
print('── Validación final ────────────────────────────────────────────────────')
suma_orig  = df_agg['Monto Gasto Normalizado'].sum()
suma_panel = df_panel['Gasto_Normalizado'].sum()
print(f'✓ Suma original  : ${suma_orig:>16,.2f}')
print(f'✓ Suma en panel  : ${suma_panel:>16,.2f}')
print(f'✓ Diferencia     : ${abs(suma_orig-suma_panel):>16,.2f}  (debe ser 0.00)')

print(f'\n── Resumen del panel ───────────────────────────────────────────────────')
print(f'  Filas totales            : {len(df_panel):,}')
print(f'  Centros gestores         : {df_panel["Centro Gestor"].nunique()}')
print(f'  Partidas presupuestales  : {df_panel["Partida Presupuestal"].nunique()}')
print(f'  Combinaciones CG×Partida : {df_panel.groupby(["Centro Gestor","Partida Presupuestal"]).ngroups}')
print(f'  Meses cubiertos          : {df_panel["Anio_Mes"].nunique()} ({df_panel["Anio_Mes"].min()} → {df_panel["Anio_Mes"].max()})')
print(f'  Meses con gasto > $0     : {(df_panel["Gasto_Normalizado"]>0).sum():,} ({(df_panel["Gasto_Normalizado"]>0).mean()*100:.1f}%)')

── Validación final ────────────────────────────────────────────────────
✓ Suma original  : $  307,895,690.17
✓ Suma en panel  : $  307,895,690.17
✓ Diferencia     : $            0.00  (debe ser 0.00)

── Resumen del panel ───────────────────────────────────────────────────
  Filas totales            : 27,200
  Centros gestores         : 36
  Partidas presupuestales  : 8
  Combinaciones CG×Partida : 272
  Meses cubiertos          : 100 (2018-01 → 2026-04)
  Meses con gasto > $0     : 10,212 (37.5%)


---
## Guardar el panel

In [60]:
cols_panel = ['Centro Gestor','Nombre Centro Gestor','Partida Presupuestal',
              'Anio','Mes','Anio_Mes','Gasto_Normalizado','Gasto_Original',
              'Num_Registros','Num_Viajes']

df_panel[cols_panel].to_parquet('panel_cg_partida_mes.parquet', index=False)
print('Guardado: panel_cg_partida_mes.parquet')
print('\nEstructura del archivo:')
pd.DataFrame({
    'Columna': cols_panel,
    'Tipo': [str(df_panel[c].dtype) for c in cols_panel],
    'Descripción': [
        'Código del centro gestor (unidad de control presupuestal)',
        'Nombre legible de la unidad administrativa',
        'Clave de partida presupuestal (375xx, 261xx, etc.)',
        'Año del gasto (Fecha Contable)',
        'Mes del gasto (1–12)',
        'Año-Mes como período — índice de la serie de tiempo',
        'VARIABLE OBJETIVO: gasto en tarifas actuales (Periodo 3)',
        'Gasto en moneda histórica (referencia para auditoría)',
        'Cantidad de renglones SAP en el mes',
        'Cantidad de viajes distintos registrados en el mes',
    ]
})

Guardado: panel_cg_partida_mes.parquet

Estructura del archivo:


,Columna,Tipo,Descripción
0,Centro Gestor,int64,Código del centro gestor (unidad de control pr...
1,Nombre Centro Gestor,object,Nombre legible de la unidad administrativa
2,Partida Presupuestal,int64,"Clave de partida presupuestal (375xx, 261xx, e..."
3,Anio,int64,Año del gasto (Fecha Contable)
4,Mes,int64,Mes del gasto (1–12)
5,Anio_Mes,period[M],Año-Mes como período — índice de la serie de t...
6,Gasto_Normalizado,float64,VARIABLE OBJETIVO: gasto en tarifas actuales (...
7,Gasto_Original,float64,Gasto en moneda histórica (referencia para aud...
8,Num_Registros,int32,Cantidad de renglones SAP en el mes
9,Num_Viajes,int32,Cantidad de viajes distintos registrados en el...


---
## Resumen ejecutivo del preprocesamiento

| Paso | Transformación | Resultado |
|---|---|---|
| Normalización de tarifas | Montos históricos → equivalentes en tarifa 2025 (Periodo 3) | 214,521 registros modificados (42.8%) |
| Mapeo CCo → Centro Gestor | 238 centros de costo → 36 centros gestores | 8 CCos imputados por prefijo (0.5% de registros) |
| Agregación mensual | Renglones por concepto → gasto total CG × Partida × Mes | 10,212 combinaciones observadas |
| Panel balanceado | Completar meses sin actividad con $0 | 27,200 filas (100 meses × 272 combinaciones) |

**Nota sobre el 62.5% de ceros:** La mayoría de los centros gestores no tienen gasto en todas las partidas todos los meses, lo cual es esperable en un entorno institucional. Este patrón será analizado en el EDA y considerado en la elección del modelo.

**Siguiente paso → A1 (EDA):** distribución de `Gasto_Normalizado`, patrones estacionales, comportamiento por partida y por centro gestor, estructura de ceros.